In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch

from torch_geometric.data import Data

In [2]:
AUG_EDGE_FILE = r"D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\augmented_graph_output\augmented_graph_edge_list.tsv"
NODE_FILE = r"D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\augmented_graph_output\augmented_graph_node_table.tsv"
CORE_DIR = r"D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\core_gene"

OUT_PYG_FILE = r"D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\augmented_graph_output\pyg_data.pt"

In [3]:
#读取文件
edge_df = pd.read_csv(AUG_EDGE_FILE, sep="\t")
node_df = pd.read_csv(NODE_FILE, sep="\t")

print(edge_df.shape)
print(node_df.shape)

display(edge_df.head())
display(node_df.head())

(346641, 11)
(3211, 6)


C:\Users\pc\AppData\Local\Temp\ipykernel_67592\1612125136.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  edge_df = pd.read_csv(AUG_EDGE_FILE, sep="\t")


,gene1,gene2,meta_r,consistency,score,edge_weight,edge_type,source,anchor_gene,partner_gene,partner_is_native_core
0,ENSG00000109819,ENSG00000132170,NaN,NaN,0.999,0.999,ppi_anchor,ppi,ENSG00000132170,ENSG00000109819,0.0
1,ENSG00000104549,ENSG00000079459,NaN,NaN,0.999,0.999,ppi_anchor,ppi,ENSG00000079459,ENSG00000104549,1.0
2,ENSG00000021762,ENSG00000141458,NaN,NaN,0.999,0.999,ppi_anchor,ppi,ENSG00000141458,ENSG00000021762,0.0
3,ENSG00000109929,ENSG00000172893,NaN,NaN,0.998,0.998,ppi_anchor,ppi,ENSG00000172893,ENSG00000109929,1.0
4,ENSG00000075239,ENSG00000112972,NaN,NaN,0.997,0.997,ppi_anchor,ppi,ENSG00000075239,ENSG00000112972,1.0


,ENSGID,SYMBOL,process,core_type,is_core_gene,is_in_meta
0,ENSG00000000938,NaN,NaN,NaN,0,1
1,ENSG00000001561,NaN,NaN,NaN,0,1
2,ENSG00000001626,NaN,NaN,NaN,0,1
3,ENSG00000003096,NaN,NaN,NaN,0,1
4,ENSG00000003137,NaN,NaN,NaN,0,1


In [4]:
#读取四个过程 core 文件，构建标签表
process_map = {
    "Biosynthesis": 0,
    "Esterification": 1,
    "Excretion": 2,
    "Uptake": 3
}

core_files = {
    "Biosynthesis": "core_gene_Biosynthesis.csv",
    "Esterification": "core_gene_Esterification.csv",
    "Excretion": "core_gene_Excretion.csv",
    "Uptake": "core_gene_Uptake.csv",
}

core_list = []
for proc, fn in core_files.items():
    df = pd.read_csv(os.path.join(CORE_DIR, fn))
    df = df.iloc[:, :2].copy()
    df.columns = ["ENSGID", "SYMBOL"]
    df["ENSGID"] = df["ENSGID"].astype(str).str.strip().str.split(".").str[0]
    df["SYMBOL"] = df["SYMBOL"].astype(str).str.strip()
    df["process"] = proc
    df["label"] = process_map[proc]
    core_list.append(df)

core_all = pd.concat(core_list, axis=0, ignore_index=True)
display(core_all.head())
print(core_all.shape)

,ENSGID,SYMBOL,process,label
0,ENSG00000116133,DHCR24,Biosynthesis,0
1,ENSG00000148377,IDI2,Biosynthesis,0
2,ENSG00000067064,IDI1,Biosynthesis,0
3,ENSG00000149809,TM7SF2,Biosynthesis,0
4,ENSG00000172893,DHCR7,Biosynthesis,0


(39, 4)


In [5]:
###处理重叠 core gene
core_count = core_all.groupby("ENSGID")["process"].nunique().reset_index()
core_count.columns = ["ENSGID", "n_process"]

core_all2 = core_all.merge(core_count, on="ENSGID", how="left")

hard_seed_df = core_all2[core_all2["n_process"] == 1].copy()
ambiguous_seed_df = core_all2[core_all2["n_process"] > 1].copy()

print("hard seeds:", hard_seed_df["ENSGID"].nunique())
print("ambiguous seeds:", ambiguous_seed_df["ENSGID"].nunique())

display(ambiguous_seed_df.head())

hard seeds: 39
ambiguous seeds: 0


,ENSGID,SYMBOL,process,label,n_process


In [6]:
#为节点建立索引
node_df = node_df.copy()
node_df["ENSGID"] = node_df["ENSGID"].astype(str).str.strip().str.split(".").str[0]

# 节点顺序
node_df = node_df.sort_values("ENSGID").reset_index(drop=True)

# ENSG -> index
gene2idx = {g: i for i, g in enumerate(node_df["ENSGID"])}
idx2gene = {i: g for g, i in gene2idx.items()}

num_nodes = len(node_df)
print("num_nodes =", num_nodes)

num_nodes = 3211


In [7]:
#构建 edge_index
edge_df = edge_df.copy()
edge_df["gene1"] = edge_df["gene1"].astype(str).str.strip().str.split(".").str[0]
edge_df["gene2"] = edge_df["gene2"].astype(str).str.strip().str.split(".").str[0]

# 只保留节点表中存在的边
edge_df = edge_df[
    edge_df["gene1"].isin(gene2idx) &
    edge_df["gene2"].isin(gene2idx)
].copy()

print("filtered edges:", edge_df.shape[0])

src = edge_df["gene1"].map(gene2idx).values
dst = edge_df["gene2"].map(gene2idx).values

# 双向展开
edge_index = torch.tensor(
    np.vstack([
        np.concatenate([src, dst]),
        np.concatenate([dst, src])
    ]),
    dtype=torch.long
)

print(edge_index.shape)

filtered edges: 346641
torch.Size([2, 693282])


In [8]:
#构建边特征
edge_type_map = {
    "meta": 0,
    "ppi_anchor": 1
}

edge_df["edge_weight"] = pd.to_numeric(edge_df["edge_weight"], errors="coerce").fillna(0.0)
edge_df["edge_type_id"] = edge_df["edge_type"].map(edge_type_map).fillna(-1).astype(int)

w = edge_df["edge_weight"].values
t = edge_df["edge_type_id"].values

# 双向展开
edge_weight = torch.tensor(np.concatenate([w, w]), dtype=torch.float32)
edge_type = torch.tensor(np.concatenate([t, t]), dtype=torch.long)

# edge_attr: [edge_weight, edge_type]
edge_attr = torch.tensor(
    np.vstack([
        np.concatenate([w, w]),
        np.concatenate([t, t]).astype(float)
    ]).T,
    dtype=torch.float32
)

print(edge_weight.shape, edge_type.shape, edge_attr.shape)

torch.Size([693282]) torch.Size([693282]) torch.Size([693282, 2])


In [9]:
# 构建节点特征x
node_df["is_core_gene"] = pd.to_numeric(node_df["is_core_gene"], errors="coerce").fillna(0).astype(int)
node_df["is_in_meta"] = pd.to_numeric(node_df["is_in_meta"], errors="coerce").fillna(0).astype(int)

node_df["core_type"] = node_df["core_type"].fillna("")
node_df["process"] = node_df["process"].fillna("")

node_df["is_native_core"] = node_df["core_type"].str.contains("native_core").astype(int)
node_df["is_anchored_core"] = node_df["core_type"].str.contains("anchored_core").astype(int)

# 四过程 multi-hot
for p in process_map.keys():
    node_df[f"proc_{p}"] = node_df["process"].str.contains(p).astype(int)

# 度和加权度
deg = np.zeros(num_nodes, dtype=float)
wdeg = np.zeros(num_nodes, dtype=float)

for s, d, ww in zip(src, dst, w):
    deg[s] += 1
    deg[d] += 1
    wdeg[s] += ww
    wdeg[d] += ww

node_df["degree"] = deg
node_df["weighted_degree"] = wdeg

# 简单标准化
def zscore(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.std() == 0:
        return np.zeros_like(arr)
    return (arr - arr.mean()) / arr.std()

node_df["degree_z"] = zscore(node_df["degree"])
node_df["weighted_degree_z"] = zscore(node_df["weighted_degree"])

feature_cols = [
    "is_core_gene",
    "is_in_meta",
    "is_native_core",
    "is_anchored_core",
    "proc_Biosynthesis",
    "proc_Esterification",
    "proc_Excretion",
    "proc_Uptake",
    "degree_z",
    "weighted_degree_z"
]

x = torch.tensor(node_df[feature_cols].values, dtype=torch.float32)
print(x.shape)
display(node_df[feature_cols].head())

torch.Size([3211, 10])


,is_core_gene,is_in_meta,is_native_core,is_anchored_core,proc_Biosynthesis,proc_Esterification,proc_Excretion,proc_Uptake,degree_z,weighted_degree_z
0,0,1,0,0,0,0,0,0,0.468955,0.468747
1,0,1,0,0,0,0,0,0,-0.602119,-0.593738
2,0,1,0,0,0,0,0,0,1.925931,2.107771
3,0,1,0,0,0,0,0,0,-0.838385,-0.800122
4,0,1,0,0,0,0,0,0,-0.846261,-0.807621


In [10]:
#构建标签 y 和训练 mask
# 初始化
y = torch.full((num_nodes,), -1, dtype=torch.long)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
seed_weight = torch.zeros(num_nodes, dtype=torch.float32)

# hard seed 去重
hard_seed_unique = hard_seed_df[["ENSGID", "label"]].drop_duplicates().copy()

hard_seed_map = dict(zip(hard_seed_unique["ENSGID"], hard_seed_unique["label"]))

for gene, label in hard_seed_map.items():
    if gene in gene2idx:
        idx = gene2idx[gene]
        y[idx] = int(label)
        train_mask[idx] = True

        core_type_val = str(node_df.loc[idx, "core_type"])
        if "native_core" in core_type_val:
            seed_weight[idx] = 1.0
        elif "anchored_core" in core_type_val:
            seed_weight[idx] = 0.6
        else:
            seed_weight[idx] = 1.0

print("num train seeds:", int(train_mask.sum()))
print("label counts among train seeds:")
for k, v in process_map.items():
    print(k, int(((y == v) & train_mask).sum()))

num train seeds: 39
label counts among train seeds:
Biosynthesis 15
Esterification 6
Excretion 10
Uptake 8


In [11]:
#构建 Data 对象
data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    edge_weight=edge_weight,
    y=y
)

data.train_mask = train_mask
data.seed_weight = seed_weight
data.edge_type = edge_type

print(data)

Data(x=[3211, 10], edge_index=[2, 693282], edge_attr=[693282, 2], y=[3211], edge_weight=[693282], train_mask=[3211], seed_weight=[3211], edge_type=[693282])


In [12]:
aux_info = {
    "gene2idx": gene2idx,
    "idx2gene": idx2gene,
    "process_map": process_map,
    "feature_cols": feature_cols
}

torch.save({
    "data": data,
    "node_df": node_df,
    "edge_df": edge_df,
    "hard_seed_df": hard_seed_df,
    "ambiguous_seed_df": ambiguous_seed_df,
    "aux_info": aux_info
}, OUT_PYG_FILE)

print("saved to:", OUT_PYG_FILE)

saved to: D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\augmented_graph_output\pyg_data.pt


In [13]:
print("x shape:", data.x.shape)
print("edge_index shape:", data.edge_index.shape)
print("edge_attr shape:", data.edge_attr.shape)
print("train seeds:", int(data.train_mask.sum()))
print("unlabeled nodes:", int((data.y == -1).sum()))

x shape: torch.Size([3211, 10])
edge_index shape: torch.Size([2, 693282])
edge_attr shape: torch.Size([693282, 2])
train seeds: 39
unlabeled nodes: 3172
